# Notebook 3 — Genetic Algorithm

Runs the existing GA from `utils/GA.py`.
No changes made to any existing code.

## Step 0 — Imports & Data Loading

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
for mod in list(sys.modules.keys()):
    if 'utils' in mod:
        del sys.modules[mod]

from utils.data_loader import (
    load_exams, load_students, load_rooms, load_timeslots, load_conflict_matrix
)
from utils.GA import GeneticExamScheduler

exams      = load_exams(path='Data/data.csv')
students   = load_students(path='Data/IDs.csv')
rooms      = load_rooms(path='Data/rooms.csv')
timeslots  = load_timeslots()
cm         = load_conflict_matrix(path='outputs/conflict_matrix.pkl')

print(f'Problem: {len(exams)} exams, {len(rooms)} rooms, {len(timeslots)} slots')

---
## Step 1 — Run GA

In [ ]:
import time

scheduler = GeneticExamScheduler(
    exams=exams,
    rooms=rooms,
    timeslots=timeslots,
    conflict_matrix=cm,
    population_size=80,
    generations=100,
    mutation_rate=0.08,
    crossover_rate=0.85,
    elite_size=4,
    tournament_size=4,
    seed=42,
)

print('Starting GA...')
t0 = time.time()
best_chromosome, best_result, history = scheduler.evolve(verbose=True)
elapsed = time.time() - t0

print(f'\nGA finished in {elapsed:.1f}s')
print(f'Fitness (higher=better) : {best_result.fitness}')
print(f'Hard violations         : {best_result.hard_violations}')
print(f'Soft violations         : {best_result.soft_violations}')

---
## Step 2 — Violation Breakdown

In [ ]:
print('BREAKDOWN')
print(f'  Room double-bookings        : {best_result.room_double_bookings}')
print(f'  Student clashes             : {best_result.student_clashes}')
print(f'  Room capacity violations    : {best_result.room_capacity_violations}')
print(f'  Same-day student clashes    : {best_result.same_day_student_clashes}')
print(f'  Uneven day distribution     : {best_result.uneven_day_distribution}')

---
## Step 3 — Convergence Plot

In [ ]:
try:
    import matplotlib.pyplot as plt

    gens = [h['generation'] for h in history]
    plt.figure(figsize=(10, 4))
    plt.plot(gens, [h['best_fitness'] for h in history], label='Best fitness', lw=2)
    plt.plot(gens, [h['average_fitness'] for h in history], label='Average fitness', lw=1, alpha=0.7)
    plt.xlabel('Generation')
    plt.ylabel('Fitness')
    plt.title('GA Convergence')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
except ImportError:
    print('matplotlib not installed -- install with: pip install matplotlib')

---
## Step 4 — Export Schedule

In [ ]:
import csv
os.makedirs('outputs', exist_ok=True)

rows = scheduler.export_schedule_rows(best_chromosome)
out_path = 'outputs/ga_schedule.csv'

with open(out_path, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)

print(f'Exported to {out_path}  ({len(rows)} rows)')
print('\nSample:')
for r in rows[:5]:
    print(f'  {r["exam"]:12s} | Room {r["room"]:5s} ({r["students in room"]:2d} students) | {r["date"]} {r["time"]}')